# 02 - Understanding XCP-D outputs

This tutorial explains the outputs produced by **XCP-D**, a postprocessing pipeline for resting-state fMRI data. XCP-D usually starts with preprocessed BOLD data from fMRIPrep and produces data that are ready for denoising-aware time-series and connectivity analyses.

By the end, you should be able to:

- recognize the XCP-D derivatives folder;
- distinguish denoised BOLD images, parcellated time series, confounds, and design files;
- inspect XCP-D outputs with Python and pandas;
- understand which files to use for a downstream connectivity analysis.

> **Important:** XCP-D outputs depend on the preprocessing and denoising options used when it was run. Always inspect the pipeline description, confounds, and QC report before interpreting a result.

## Before you begin

Set `xcpd_dir` below to your XCP-D derivatives directory. A common layout is:

```text
derivatives/
└── xcp_d/
    ├── dataset_description.json
    ├── sub-01/
    │   └── func/
    └── sub-02/
```

The discovery cells are safe to run without data: they print a message if the directory is not available.

In [1]:
from pathlib import Path

import pandas as pd

# Change this path to the XCP-D derivatives directory for your dataset.
xcpd_dir = Path("/projects/aabdulrasul/Tutorials/derivatives/xcp_d")

if xcpd_dir.exists():
    print(f"Using: {xcpd_dir.resolve()}")
else:
    print(f"No directory found at {xcpd_dir.resolve()}.")
    print("Update xcpd_dir when you are ready to inspect your own outputs.")

Using: /mnt/tigrlab/projects/aabdulrasul/Tutorials/derivatives/xcp_d


In [2]:
# List the subjects that XCP-D found.
subject_dirs = []
if xcpd_dir.exists():
    for item in xcpd_dir.iterdir():
        if item.is_dir() and item.name.startswith("sub-"):
            subject_dirs.append(item)

if subject_dirs:
    print(f"Found {len(subject_dirs)} subject(s):")
    for subject_dir in sorted(subject_dirs):
        print(f"  {subject_dir.name}")
else:
    print("No subject directories found yet. Check xcpd_dir above.")

Found 3 subject(s):
  sub-CMH00000001
  sub-CMH00000003
  sub-CMH00000005


In [3]:
def list_files(folder: Path):
    """Return all files below a folder using simple Path operations."""
    files = []
    for item in folder.iterdir():
        if item.is_dir():
            files.extend(list_files(item))
        elif item.is_file():
            files.append(item)
    return files


all_files = list_files(xcpd_dir) if xcpd_dir.exists() else []
print(f"Found {len(all_files)} files.")
for path in sorted(all_files)[:12]:
    print(path)

Found 2832 files.
/projects/aabdulrasul/Tutorials/derivatives/xcp_d/sub-CMH00000001/log/20260313-115607_38a55dac-95a8-46bf-93cd-0364aca20585/xcp_d.toml
/projects/aabdulrasul/Tutorials/derivatives/xcp_d/sub-CMH00000001/ses-01/anat/sub-CMH00000001_ses-01_hemi-L_space-fsLR_den-32k_desc-hcp_inflated.surf.gii
/projects/aabdulrasul/Tutorials/derivatives/xcp_d/sub-CMH00000001/ses-01/anat/sub-CMH00000001_ses-01_hemi-L_space-fsLR_den-32k_desc-hcp_midthickness.surf.gii
/projects/aabdulrasul/Tutorials/derivatives/xcp_d/sub-CMH00000001/ses-01/anat/sub-CMH00000001_ses-01_hemi-L_space-fsLR_den-32k_desc-hcp_vinflated.surf.gii
/projects/aabdulrasul/Tutorials/derivatives/xcp_d/sub-CMH00000001/ses-01/anat/sub-CMH00000001_ses-01_hemi-L_space-fsLR_den-32k_pial.surf.gii
/projects/aabdulrasul/Tutorials/derivatives/xcp_d/sub-CMH00000001/ses-01/anat/sub-CMH00000001_ses-01_hemi-L_space-fsLR_den-32k_white.surf.gii
/projects/aabdulrasul/Tutorials/derivatives/xcp_d/sub-CMH00000001/ses-01/anat/sub-CMH00000001_ses-

## 1. How to read an XCP-D filename

XCP-D filenames preserve the BIDS entities that identify the acquisition and add entities that describe the processing. A typical functional filename might look like:

```text
sub-01_task-rest_run-01_space-MNI152NLin2009cAsym_desc-denoised_bold.nii.gz
```

Read it from left to right:

- `sub-01`: participant;
- `task-rest`: resting-state acquisition;
- `run-01`: run number;
- `space-MNI...`: coordinate space;
- `desc-denoised_bold`: a denoised BOLD image;
- `.nii.gz`: compressed NIfTI image.

Other XCP-D outputs may use `timeseries`, `confounds`, `motion`, `design`, or `qc` in the filename. The exact set depends on the XCP-D version and command-line options.

In [4]:
print("Denoised BOLD images:")
denoised_bold_files = [path for path in all_files if "desc-denoised_bold" in path.name]
for path in sorted(denoised_bold_files):
    print(path)
if not denoised_bold_files:
    print("No denoised BOLD files found.")

print("\nParcellated or regional time series:")
timeseries_files = [path for path in all_files if "timeseries" in path.name]
for path in sorted(timeseries_files):
    print(path)
if not timeseries_files:
    print("No time-series files found.")

Denoised BOLD images:
/projects/aabdulrasul/Tutorials/derivatives/xcp_d/sub-CMH00000001/ses-01/func/sub-CMH00000001_ses-01_task-nback_run-01_space-fsLR_den-91k_desc-denoised_bold.dtseries.nii
/projects/aabdulrasul/Tutorials/derivatives/xcp_d/sub-CMH00000001/ses-01/func/sub-CMH00000001_ses-01_task-nback_run-01_space-fsLR_den-91k_desc-denoised_bold.json
/projects/aabdulrasul/Tutorials/derivatives/xcp_d/sub-CMH00000001/ses-01/func/sub-CMH00000001_ses-01_task-nback_run-02_space-fsLR_den-91k_desc-denoised_bold.dtseries.nii
/projects/aabdulrasul/Tutorials/derivatives/xcp_d/sub-CMH00000001/ses-01/func/sub-CMH00000001_ses-01_task-nback_run-02_space-fsLR_den-91k_desc-denoised_bold.json
/projects/aabdulrasul/Tutorials/derivatives/xcp_d/sub-CMH00000001/ses-01/func/sub-CMH00000001_ses-01_task-rest_run-01_space-fsLR_den-91k_desc-denoised_bold.dtseries.nii
/projects/aabdulrasul/Tutorials/derivatives/xcp_d/sub-CMH00000001/ses-01/func/sub-CMH00000001_ses-01_task-rest_run-01_space-fsLR_den-91k_desc-den

## 2. The main XCP-D outputs

### Denoised BOLD images

A `desc-denoised_bold.nii.gz` file is a 4D image with nuisance variation reduced according to the selected XCP-D workflow. It remains a voxelwise image, so it is useful for visualization, masks, and analyses that need spatial information.

Do not assume that every XCP-D run used the same denoising strategy. Check the command, pipeline description, and accompanying metadata before comparing outputs.

### Regional or parcellated time series

A `timeseries.tsv` or `timeseries.csv` file contains rows representing time points and columns representing regions, parcels, or other signals. This is often the most convenient input for functional connectivity analyses. Each file should be matched to the correct subject, task, run, space, and atlas.

In [5]:
def read_table(path: Path) -> pd.DataFrame:
    if path.suffix == ".tsv":
        return pd.read_csv(path, sep="\t")
    return pd.read_csv(path)


if timeseries_files:
    timeseries_file = sorted(timeseries_files)[0]
    timeseries = read_table(timeseries_file)
    print(f"Example: {timeseries_file}")
    print(f"Shape: {timeseries.shape[0]} time points x {timeseries.shape[1]} columns")
    display(timeseries.head())
else:
    print("No XCP-D time-series table found yet. Update xcpd_dir and rerun this cell.")

ParserError: Error tokenizing data. C error: Expected 1 fields in line 3, saw 2


## 3. Confounds and design files

XCP-D may write tables describing the regressors used during denoising and the resulting design matrix. Common contents include motion estimates, framewise displacement, censoring or outlier indicators, and nuisance components.

- A **confounds table** helps you inspect the nuisance signals and identify high-motion volumes.
- A **design matrix** shows the regressors that were actually used in a model or denoising step.
- A **motion file** summarizes head movement over time.

The important question is not simply whether a file exists, but whether it matches the BOLD or timeseries file you are analyzing. Match subject, task, run, space, atlas, and number of time points.

In [ ]:
print("Confounds:")
confounds_files = [path for path in all_files if "confounds" in path.name]
for path in sorted(confounds_files):
    print(path)
if not confounds_files:
    print("No confounds files found.")

print("\nDesign matrices:")
design_files = [path for path in all_files if "design" in path.name]
for path in sorted(design_files):
    print(path)
if not design_files:
    print("No design files found.")

print("\nMotion summaries:")
motion_files = [path for path in all_files if "motion" in path.name]
for path in sorted(motion_files):
    print(path)
if not motion_files:
    print("No motion files found.")

## 4. Quality control and provenance

Before using XCP-D outputs, review the subject-level QC report if one is available. Look for excessive motion, censoring that removes many volumes, poor brain coverage, registration problems, and unexpected missing data.

Also inspect `dataset_description.json`, HTML reports, and any boilerplate or command-line record saved with the derivatives. These documents tell you which XCP-D version and options produced the files. Those details are part of the scientific meaning of the output.

In [ ]:
print("QC reports:")
html_files = [path for path in all_files if path.suffix == ".html"]
for path in sorted(html_files):
    print(path)
if not html_files:
    print("No HTML QC reports found.")

print("\nPipeline description:")
description_files = [path for path in all_files if path.name == "dataset_description.json"]
for path in sorted(description_files):
    print(path)
if not description_files:
    print("No dataset description found.")

## 5. Choosing an output for analysis

| Goal | Start with |
| --- | --- |
| Inspect denoised voxelwise data | matching `desc-denoised_bold.nii.gz` |
| Compute parcel-to-parcel connectivity | matching atlas-specific `timeseries.tsv` or `timeseries.csv` |
| Investigate motion or censoring | matching confounds and motion tables |
| Reproduce or explain denoising | design matrix, confounds, metadata, and pipeline description |
| Check data quality | XCP-D HTML/QC reports and visual inspection |

For connectivity, use timeseries files from the same atlas and compatible space. Do not concatenate runs until you have checked that their columns represent the same regions and that the time points are in the expected order.

In [ ]:
if timeseries_files:
    numeric_timeseries = timeseries.select_dtypes(include="number")
    print(f"Numeric signal columns: {numeric_timeseries.shape[1]}")
    print("Missing values per column (first 10):")
    display(numeric_timeseries.isna().sum().head(10))
    print("\nCorrelation matrix preview:")
    display(numeric_timeseries.corr().iloc[:5, :5])
else:
    print("Once a timeseries file is available, this cell checks missing values and previews correlations.")